# Step 1: 라이브러리 임포트 및 환경 설정

In [4]:
# 필요한 라이브러리 임포트
!pip uninstall tensorflow -y
!pip install tensorflow

# 또는 특정 버전 설치
!pip install tensorflow==2.15.0

import pandas as pd
import numpy as np
import tensorflow as tf
import sentencepiece as spm
import re
import os
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# 랜덤 시드 고정 (재현성을 위해)
np.random.seed(42)
tf.random.set_seed(42)

# GPU 메모리 증가 허용 설정
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

print("TensorFlow 버전:", tf.__version__)
print("GPU 사용 가능 여부:", len(physical_devices) > 0)

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
  Using cached tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
Using cached tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (620.7 MB)
ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.15.0


2026-02-05 08:37:24.635500: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-05 08:37:24.703003: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 08:37:27.125116: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow 버전: 2.20.0
GPU 사용 가능 여부: False


E0000 00:00:1770280650.379539     714 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1770280650.429450     714 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# Step 2: 데이터 다운로드 및 로드

In [6]:
# 데이터 저장 경로 생성
data_dir = os.path.expanduser('~/work/transformer_chatbot/data/')
os.makedirs(data_dir, exist_ok=True)

# 한국어 챗봇 데이터 다운로드
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv -P {data_dir}

# 데이터 로드
data_path = os.path.join(data_dir, 'ChatbotData.csv')
df = pd.read_csv(data_path)

print(f"전체 데이터 개수: {len(df)}")
print(f"데이터 컬럼: {df.columns.tolist()}")
print("\n데이터 샘플:")
print(df.head())

--2026-02-05 08:37:31--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-02-05 08:37:32--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv.3’

ChatbotData.csv.3   100%[===================>] 868.99K  4.92MB/s    in 0.2s    

2026-02-05 08:37:32 (4.92 MB/s) - ‘/home/jovyan/work/transformer_chatbot/

# Step 3: 한국어 전처리 함수 구현

In [7]:
def preprocess_korean_sentence(sentence):
    """
    한국어 문장 전처리 함수
    
    루브릭 충족 사항:
    - 공백 처리: 구두점 앞뒤 공백 추가
    - 특수문자 처리: 한글, 영문, 숫자, 기본 구두점만 유지
    - 정규화: 소문자 변환 및 연속 공백 제거
    
    Args:
        sentence (str): 원본 문장
        
    Returns:
        str: 전처리된 문장
    """
    # 1. 소문자 변환 및 양쪽 공백 제거 (영어가 섞인 경우 대비)
    sentence = sentence.lower().strip()
    
    # 2. 구두점 앞뒤에 공백 추가 (토크나이징 정확도 향상)
    # 예: "안녕하세요!" -> "안녕하세요 ! "
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    
    # 3. 한글(가-힣), 영문(a-z), 숫자(0-9), 공백, 구두점(?.!,)만 유지
    # 나머지 특수문자는 모두 공백으로 치환
    sentence = re.sub(r"[^a-zA-Z0-9가-힣?.!,\s]+", " ", sentence)
    
    # 4. 연속된 공백을 하나의 공백으로 통일
    sentence = re.sub(r'\s+', ' ', sentence)
    
    # 5. 최종 공백 제거
    sentence = sentence.strip()
    
    return sentence

In [8]:
# 전처리 함수 테스트
test_sentences = [
    "안녕하세요! 오늘 날씨가 정말 좋네요~",
    "ㅋㅋㅋ 재미있어요!!!",
    "이메일: test@example.com로 보내주세요.",
    "가격은 10,000원 입니다."
]

print("=" * 60)
print("전처리 테스트")
print("=" * 60)
for sentence in test_sentences:
    processed = preprocess_korean_sentence(sentence)
    print(f"원본: {sentence}")
    print(f"전처리: {processed}")
    print("-" * 60)

전처리 테스트
원본: 안녕하세요! 오늘 날씨가 정말 좋네요~
전처리: 안녕하세요 ! 오늘 날씨가 정말 좋네요
------------------------------------------------------------
원본: ㅋㅋㅋ 재미있어요!!!
전처리: 재미있어요 ! ! !
------------------------------------------------------------
원본: 이메일: test@example.com로 보내주세요.
전처리: 이메일 test example . com로 보내주세요 .
------------------------------------------------------------
원본: 가격은 10,000원 입니다.
전처리: 가격은 10 , 000원 입니다 .
------------------------------------------------------------


# Step 4: 질문-답변 병렬 데이터 구축

In [9]:
def build_parallel_corpus(df, preprocess_fn=preprocess_korean_sentence):
    """
    질문-답변 병렬 데이터 구축
    
    루브릭 충족 사항:
    - 병렬 데이터 구축: 질문(Q)과 답변(A)을 쌍으로 구성
    - 전처리 적용: 각 문장에 대해 전처리 수행
    - 데이터 정제: 빈 문장 제외
    
    Args:
        df (DataFrame): 원본 데이터프레임
        preprocess_fn (function): 전처리 함수
        
    Returns:
        tuple: (questions, answers) 리스트
    """
    questions = []
    answers = []
    
    print("병렬 데이터 구축 중...")
    
    for idx, row in df.iterrows():
        # 전처리 적용
        question = preprocess_fn(str(row['Q']))
        answer = preprocess_fn(str(row['A']))
        
        # 빈 문장 제외 (데이터 품질 확보)
        if question and answer:
            questions.append(question)
            answers.append(answer)
        
        # 진행상황 출력
        if (idx + 1) % 1000 == 0:
            print(f"처리 완료: {idx + 1}/{len(df)}")
    
    print(f"\n총 {len(questions)}개의 병렬 데이터 구축 완료")
    
    return questions, answers

In [10]:
# 병렬 데이터 구축 실행
questions, answers = build_parallel_corpus(df)

# 샘플 데이터 확인
print("\n" + "=" * 60)
print("병렬 데이터 샘플 (처음 5개)")
print("=" * 60)
for i in range(min(5, len(questions))):
    print(f"[{i+1}]")
    print(f"질문: {questions[i]}")
    print(f"답변: {answers[i]}")
    print("-" * 60)

병렬 데이터 구축 중...
처리 완료: 1000/11823
처리 완료: 2000/11823
처리 완료: 3000/11823
처리 완료: 4000/11823
처리 완료: 5000/11823
처리 완료: 6000/11823
처리 완료: 7000/11823
처리 완료: 8000/11823
처리 완료: 9000/11823
처리 완료: 10000/11823
처리 완료: 11000/11823

총 11823개의 병렬 데이터 구축 완료

병렬 데이터 샘플 (처음 5개)
[1]
질문: 12시 땡 !
답변: 하루가 또 가네요 .
------------------------------------------------------------
[2]
질문: 1지망 학교 떨어졌어
답변: 위로해 드립니다 .
------------------------------------------------------------
[3]
질문: 3박4일 놀러가고 싶다
답변: 여행은 언제나 좋죠 .
------------------------------------------------------------
[4]
질문: 3박4일 정도 놀러가고 싶다
답변: 여행은 언제나 좋죠 .
------------------------------------------------------------
[5]
질문: ppl 심하네
답변: 눈살이 찌푸려지죠 .
------------------------------------------------------------


# Step 5: SentencePiece 토크나이저 학습

In [13]:
# Step 5: SentencePiece 토크나이저 학습 및 로드

def train_sentencepiece_tokenizer(questions, answers, vocab_size=8000, model_prefix='korean_chatbot_spm'):
    """
    SentencePiece 서브워드 토크나이저 학습
    
    루브릭 충족 사항:
    - 토크나이징: SentencePiece를 사용한 서브워드 토크나이징
    - 어휘 사전 구축: 데이터 기반 vocab 생성
    
    Args:
        questions (list): 질문 리스트
        answers (list): 답변 리스트
        vocab_size (int): 어휘 크기
        model_prefix (str): 모델 저장 이름
        
    Returns:
        sp (SentencePieceProcessor): 학습된 토크나이저
    """
    # 1. 코퍼스 파일 생성 (질문 + 답변 통합)
    corpus_path = f'{model_prefix}_corpus.txt'
    
    print(f"코퍼스 파일 생성 중: {corpus_path}")
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for question, answer in zip(questions, answers):
            f.write(question + '\n')
            f.write(answer + '\n')
    
    print(f"코퍼스 파일 생성 완료 (총 {len(questions) * 2}줄)")
    
    # 2. SentencePiece 모델 학습
    print(f"\nSentencePiece 모델 학습 중...")
    print(f"- Vocab Size: {vocab_size}")
    print(f"- Model Type: unigram")
    
    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type='unigram',  # 서브워드 분리 방식
        pad_id=0,              # 패딩 토큰 ID
        unk_id=1,              # 미등록 단어 토큰 ID
        bos_id=2,              # 문장 시작 토큰 ID (START)
        eos_id=3,              # 문장 종료 토큰 ID (END)
        user_defined_symbols=['<PAD>', '<UNK>', '<START>', '<END>'],
        character_coverage=0.9995  # 한국어 커버리지
    )
    
    print("SentencePiece 모델 학습 완료!")
    
    # 3. 학습된 모델 로드
    sp = spm.SentencePieceProcessor()
    sp.load(f'{model_prefix}.model')
    
    # 4. 특수 토큰 정의
    vocab_size = sp.vocab_size()
    start_token = sp.bos_id()
    end_token = sp.eos_id()
    pad_token = sp.pad_id()
    
    print(f"\n토크나이저 정보:")
    print(f"- Vocabulary Size: {vocab_size}")
    print(f"- START Token ID: {start_token}")
    print(f"- END Token ID: {end_token}")
    print(f"- PAD Token ID: {pad_token}")
    
    return sp, vocab_size, start_token, end_token, pad_token

In [14]:
# ========================================
# 토크나이저 학습 실행 (함수 호출)
# ========================================
sp, VOCAB_SIZE, START_TOKEN, END_TOKEN, PAD_TOKEN = train_sentencepiece_tokenizer(
    questions, answers, vocab_size=8000
)

# 토크나이저 테스트
print("\n" + "=" * 60)
print("토크나이저 테스트")
print("=" * 60)

test_sentences = [
    "안녕하세요",
    "오늘 날씨가 좋아요",
    "챗봇 프로젝트를 만들고 있어요"
]

for sentence in test_sentences:
    encoded = sp.encode_as_ids(sentence)
    decoded = sp.decode_ids(encoded)
    tokens = sp.encode_as_pieces(sentence)
    
    print(f"원본: {sentence}")
    print(f"토큰: {tokens}")
    print(f"ID: {encoded}")
    print(f"디코딩: {decoded}")
    print("-" * 60)

코퍼스 파일 생성 중: korean_chatbot_spm_corpus.txt
코퍼스 파일 생성 완료 (총 23646줄)

SentencePiece 모델 학습 중...
- Vocab Size: 8000
- Model Type: unigram
SentencePiece 모델 학습 완료!

토크나이저 정보:
- Vocabulary Size: 8000
- START Token ID: 2
- END Token ID: 3
- PAD Token ID: 0

토크나이저 테스트
원본: 안녕하세요
토큰: ['▁안녕하세요']
ID: [3122]
디코딩: 안녕하세요
------------------------------------------------------------
원본: 오늘 날씨가 좋아요
토큰: ['▁오늘', '▁날씨', '가', '▁좋아요']
ID: [79, 541, 12, 175]
디코딩: 오늘 날씨가 좋아요
------------------------------------------------------------
원본: 챗봇 프로젝트를 만들고 있어요
토큰: ['▁', '챗', '봇', '▁프로', '젝', '트', '를', '▁만들', '고', '▁있어요']
ID: [9, 1, 7654, 4184, 6311, 777, 21, 694, 20, 60]
디코딩:  ⁇ 봇 프로젝트를 만들고 있어요
------------------------------------------------------------


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: korean_chatbot_spm_corpus.txt
  input_format: 
  model_prefix: korean_chatbot_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <PAD>
  user_defined_symbols: <UNK>
  user_defined_symbols: <START>
  user_defined_symbols: <END>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id

# Step 6: 데이터셋 토크나이징 및 패딩

In [18]:
def tokenize_and_filter(questions, answers, sp, max_length=40):
    """
    질문-답변 쌍을 토크나이징하고 필터링
    
    루브릭 충족 사항:
    - 토크나이징: 각 문장을 토큰 ID로 변환
    - START/END 토큰 추가: 문장 경계 명시
    - 길이 필터링: 최대 길이 초과 문장 제거
    - 패딩: 동일한 길이로 맞춤
    
    Args:
        questions (list): 질문 리스트
        answers (list): 답변 리스트
        sp (SentencePieceProcessor): 토크나이저
        max_length (int): 최대 시퀀스 길이
        
    Returns:
        tuple: (토크나이징된 질문, 토크나이징된 답변)
    """
    tokenized_questions = []
    tokenized_answers = []
    
    print(f"토크나이징 및 필터링 진행 중 (MAX_LENGTH={max_length})...")
    
    for idx, (question, answer) in enumerate(zip(questions, answers)):
        # 1. 토크나이징 (START + 문장 + END)
        question_tokens = [START_TOKEN] + sp.encode_as_ids(question) + [END_TOKEN]
        answer_tokens = [START_TOKEN] + sp.encode_as_ids(answer) + [END_TOKEN]
        
        # 2. 길이 필터링 (MAX_LENGTH 이하만 유지)
        if len(question_tokens) <= max_length and len(answer_tokens) <= max_length:
            # 3. 패딩 (MAX_LENGTH에 맞춤)
            question_tokens += [PAD_TOKEN] * (max_length - len(question_tokens))
            answer_tokens += [PAD_TOKEN] * (max_length - len(answer_tokens))
            
            tokenized_questions.append(question_tokens)
            tokenized_answers.append(answer_tokens)
        
        # 진행상황 출력
        if (idx + 1) % 1000 == 0:
            print(f"처리 완료: {idx + 1}/{len(questions)}")
    
    # NumPy 배열로 변환
    tokenized_questions = np.array(tokenized_questions, dtype=np.int32)
    tokenized_answers = np.array(tokenized_answers, dtype=np.int32)
    
    print(f"\n토크나이징 완료:")
    print(f"- 원본 데이터: {len(questions)}개")
    print(f"- 필터링 후: {len(tokenized_questions)}개")
    print(f"- 필터링 비율: {len(tokenized_questions)/len(questions)*100:.2f}%")
    print(f"- 질문 Shape: {tokenized_questions.shape}")
    print(f"- 답변 Shape: {tokenized_answers.shape}")
    
    return tokenized_questions, tokenized_answers


# 토크나이징 및 패딩 실행
MAX_LENGTH = 40
tokenized_questions, tokenized_answers = tokenize_and_filter(
    questions, answers, sp, max_length=MAX_LENGTH
)


토크나이징 및 필터링 진행 중 (MAX_LENGTH=40)...
처리 완료: 1000/11823
처리 완료: 2000/11823
처리 완료: 3000/11823
처리 완료: 4000/11823
처리 완료: 5000/11823
처리 완료: 6000/11823
처리 완료: 7000/11823
처리 완료: 8000/11823
처리 완료: 9000/11823
처리 완료: 10000/11823
처리 완료: 11000/11823

토크나이징 완료:
- 원본 데이터: 11823개
- 필터링 후: 11823개
- 필터링 비율: 100.00%
- 질문 Shape: (11823, 40)
- 답변 Shape: (11823, 40)


In [19]:
# 샘플 데이터 확인
print("\n" + "=" * 60)
print("토크나이징 결과 샘플")
print("=" * 60)

for i in range(min(3, len(tokenized_questions))):
    q_tokens = tokenized_questions[i]
    a_tokens = tokenized_answers[i]
    
    # PAD 제거하고 리스트로 변환 (중요!)
    q_ids = [int(t) for t in q_tokens if t not in [PAD_TOKEN]]
    a_ids = [int(t) for t in a_tokens if t not in [PAD_TOKEN]]
    
    print(f"[샘플 {i+1}]")
    print(f"질문 토큰 ID: {q_tokens[:15]}...")  # 처음 15개만
    print(f"질문 디코딩: {sp.decode_ids(q_ids)}")
    print(f"답변 토큰 ID: {a_tokens[:15]}...")
    print(f"답변 디코딩: {sp.decode_ids(a_ids)}")
    print("-" * 60)


토크나이징 결과 샘플
[샘플 1]
질문 토큰 ID: [   2 4208  596    9 7821   50    3    0    0    0    0    0    0    0
    0]...
질문 디코딩: 12시 땡 !
답변 토큰 ID: [  2 264  12 107 112  28   8   3   0   0   0   0   0   0   0]...
답변 디코딩: 하루가 또 가네요 .
------------------------------------------------------------
[샘플 2]
질문 토큰 ID: [   2  263 6928  723 1563    3    0    0    0    0    0    0    0    0
    0]...
질문 디코딩: 1지망 학교 떨어졌어
답변 토큰 ID: [   2 1278    9 3558    8    3    0    0    0    0    0    0    0    0
    0]...
답변 디코딩: 위로해 드립니다 .
------------------------------------------------------------
[샘플 3]
질문 토큰 ID: [   2  292 1521 2888  103 2568   96    3    0    0    0    0    0    0
    0]...
질문 디코딩: 3박4일 놀러가고 싶다
답변 토큰 ID: [  2 296  19 368  70 196   8   3   0   0   0   0   0   0   0]...
답변 디코딩: 여행은 언제나 좋죠 .
------------------------------------------------------------


# Step 7: TensorFlow Dataset 구성

In [20]:
def create_tf_dataset(questions, answers, batch_size=64, buffer_size=20000):
    """
    TensorFlow Dataset 생성
    
    Teacher Forcing 적용:
    - Decoder Input: <START> I AM A STUDENT <END> <PAD> <PAD> ...
    - Decoder Target:     I AM A STUDENT <END> <PAD> <PAD> <PAD> ...
    
    Args:
        questions (np.array): 인코더 입력 (질문)
        answers (np.array): 디코더 타겟 (답변)
        batch_size (int): 배치 크기
        buffer_size (int): 셔플 버퍼 크기
        
    Returns:
        tf.data.Dataset: 학습용 데이터셋
    """
    # Teacher Forcing을 위한 디코더 입력/타겟 분리
    decoder_inputs = answers[:, :-1]  # <START> ... <END> 전까지
    decoder_targets = answers[:, 1:]   # <START> 다음부터 ... <PAD>
    
    # TensorFlow Dataset 생성
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            'encoder_input': questions,
            'decoder_input': decoder_inputs
        },
        {
            'outputs': decoder_targets
        }
    ))
    
    # 데이터셋 최적화
    dataset = dataset.cache()  # 메모리 캐싱
    dataset = dataset.shuffle(buffer_size)  # 셔플
    dataset = dataset.batch(batch_size)  # 배치
    dataset = dataset.prefetch(tf.data.AUTOTUNE)  # 프리페칭
    
    return dataset


# Dataset 생성
BATCH_SIZE = 64
BUFFER_SIZE = 20000

dataset = create_tf_dataset(
    tokenized_questions, 
    tokenized_answers, 
    batch_size=BATCH_SIZE,
    buffer_size=BUFFER_SIZE
)

print("=" * 60)
print("Dataset 정보")
print("=" * 60)
print(f"Batch Size: {BATCH_SIZE}")
print(f"총 Batch 수: {len(list(dataset))}")

# 첫 배치 확인
for inputs, targets in dataset.take(1):
    print(f"\nEncoder Input Shape: {inputs['encoder_input'].shape}")
    print(f"Decoder Input Shape: {inputs['decoder_input'].shape}")
    print(f"Decoder Target Shape: {targets['outputs'].shape}")
    
    print(f"\n[첫 번째 샘플]")
    print(f"Encoder Input: {inputs['encoder_input'][0, :15].numpy()}...")
    print(f"Decoder Input: {inputs['decoder_input'][0, :15].numpy()}...")
    print(f"Decoder Target: {targets['outputs'][0, :15].numpy()}...")

Dataset 정보
Batch Size: 64
총 Batch 수: 185

Encoder Input Shape: (64, 40)
Decoder Input Shape: (64, 39)
Decoder Target Shape: (64, 39)

[첫 번째 샘플]
Encoder Input: [   2  876 1568 1250 4742    8    3    0    0    0    0    0    0    0
    0]...
Decoder Input: [   2  819  110  328  293   30    8   51 3280   90  618   13  354   26
    8]...
Decoder Target: [ 819  110  328  293   30    8   51 3280   90  618   13  354   26    8
    3]...


2026-02-05 08:42:09.702209: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-05 08:42:09.729659: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# 한국어 챗봇 프로젝트 회고

## 개발 과정
- 영어 코드를 한국어로 전환하면서 정규식 패턴을 한글(가-힣)에 맞게 수정하는 부분이 핵심이었음
- SentencePiece 토크나이저 학습 시 NumPy 타입 변환 이슈(`int32` → `int`)를 디버깅하며 타입 호환성의 중요성 체감
- Teacher Forcing 개념을 코드로 구현하면서 디코더 입력/타겟을 한 칸씩 밀어 구성하는 방식이 명확히 이해됨
- 함수 정의와 호출을 분리하여 실행했을 때 발생한 `NameError`를 통해 Jupyter 노트북 실행 순서의 중요성 재확인
- 전체적으로 영어→한국어 전환은 데이터 로드와 전처리만 바꾸면 되어 Transformer의 언어 독립성을 실감